In [ ]:
# Install required packages for pure PyTorch implementation
# Run this cell if you haven't installed the required packages

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install scikit-learn pillow matplotlib numpy
# !pip install scikit-image tqdm

print("Required packages for this notebook:")
print("- torch, torchvision, torchaudio")
print("- scikit-learn")
print("- pillow (PIL)")
print("- matplotlib")
print("- numpy")
print("- scikit-image")
print("- tqdm (for progress bars)")
print("- pandas (optional)")
print("\nTo install, uncomment and run the pip install commands above.")

In [ ]:
# Install tqdm for progress bars
try:
    from tqdm.auto import tqdm
    print("✅ tqdm already installed")
except ImportError:
    print("📦 Installing tqdm...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm.auto import tqdm
    print("✅ tqdm installed successfully!")

# Test tqdm
import time
print("\n🧪 Testing progress bar:")
for i in tqdm(range(5), desc="Testing"):
    time.sleep(0.1)
print("🎉 Progress bar working correctly!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
import numpy as np
import os
print("GPU available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name())

In [ ]:
# Thiết lập device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# ===================================================================
# FINAL CONFIGURATION - COLOR FOLDER AS REQUESTED
# ===================================================================
import torch
import random
import numpy as np

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ===== DATASET PATHS =====
DURIAN_DIR = "dataset/Durian_Leaf_Diseases"       # Durian dataset (6 classes, pre-split)
PLANTVILLAGE_DIR = "dataset/PlantVillage/color"   # PlantVillage COLOR folder (38 classes)

# ===== MODEL & TRAINING CONFIG =====
MODEL_NAME = "vit_b_16"
DATASET_NAME = "Combined_Durian_PlantVillage_Color"
BASE_FILENAME = f"{MODEL_NAME}-{DATASET_NAME.lower()}"

# Training parameters - SYNCHRONIZED WITH EFFICIENTNET & MOBILENETV3
LEARNING_RATE = 0.001
epochs = 10       # Synchronized: 20 epochs for fair comparison
batch_size = 88
IMG_SIZE = 224

# Windows-optimized settings
NUM_WORKERS = 0  
PIN_MEMORY = False

# Advanced settings
SEED = 42
ENABLE_OVERFITTING_CHECK = True
MAX_OVERFITTING_GAP = 0.15    # Synchronized: 15% overfitting gap

# Model saving
MODEL_DIR = f"./modelsCP/{BASE_FILENAME}-lr{LEARNING_RATE}"
MODEL_SAVE_PATH = f"{MODEL_DIR}/best_model.pth"

# Create model directory
import os
os.makedirs(MODEL_DIR, exist_ok=True)

print("🔧 SYNCHRONIZED CONFIGURATION (ViT ↔ EfficientNet ↔ MobileNetV3):")
print(f"  - Model: {MODEL_NAME}")
print(f"  - Dataset: {DATASET_NAME}")
print(f"  - Durian Path: {DURIAN_DIR} (train/val/test structure)")
print(f"  - PlantVillage Path: {PLANTVILLAGE_DIR} (38 class subfolders)")
print(f"  - Structure: Each subfolder in color/ = 1 class, contains images")
print(f"  - Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Epochs: {epochs} (SYNCHRONIZED), Batch Size: {batch_size}")
print(f"  - Overfitting Gap: {MAX_OVERFITTING_GAP*100:.1f}% (SYNCHRONIZED)")
print(f"  - Model Save Path: {MODEL_SAVE_PATH}")

# Set random seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\n✅ USING COLOR FOLDER - Each subfolder = 1 class with images inside")
print("🥥 Durian: Pre-split structure (train/val/test)")
print("🎨 PlantVillage Color: Direct class folders")
print("🔄 SYNCHRONIZED: 20 epochs, 15% overfitting gap for fair comparison")

This source code requires a **HIGH RAM** machine.

You might need to install this on your system:

apt-get install python3-opencv git

For PyTorch: pip install torch torchvision torchaudio

In [ ]:
# ===================================================================
# COMBINED DATASET LOADING - CLEANED UP
# ===================================================================
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import os

def load_combined_dataset():
    """Load combined Durian + PlantVillage Color dataset"""
    print("🔄 Loading Combined Dataset...")
    
    # Data transforms
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Load Durian dataset (pre-split)
    durian_train = datasets.ImageFolder(
        root=os.path.join(DURIAN_DIR, 'train'), transform=train_transform
    )
    durian_val = datasets.ImageFolder(
        root=os.path.join(DURIAN_DIR, 'val'), transform=val_transform
    )
    
    # Load PlantVillage Color dataset  
    pv_full_dataset = datasets.ImageFolder(
        root=PLANTVILLAGE_DIR, transform=val_transform
    )
    
    # Split PlantVillage: 80% train, 20% val
    pv_total = len(pv_full_dataset)
    pv_train_size = int(0.8 * pv_total)
    pv_val_size = pv_total - pv_train_size
    
    pv_train_dataset, pv_val_dataset = torch.utils.data.random_split(
        pv_full_dataset, [pv_train_size, pv_val_size],
        generator=torch.Generator().manual_seed(SEED)
    )
    
    # Apply transforms to PlantVillage splits
    class TransformDataset(Dataset):
        def __init__(self, subset, transform):
            self.subset = subset
            self.transform = transform
        
        def __getitem__(self, index):
            x, y = self.subset[index]
            if self.transform:
                if isinstance(x, torch.Tensor):
                    x = transforms.ToPILImage()(x)
                x = self.transform(x)
            return x, y
        
        def __len__(self):
            return len(self.subset)
    
    pv_train_transformed = TransformDataset(pv_train_dataset, train_transform)
    pv_val_transformed = TransformDataset(pv_val_dataset, val_transform)
    
    # Combine datasets with correct class indexing
    class CombinedDataset(Dataset):
        def __init__(self, durian_dataset, pv_dataset, pv_class_offset):
            self.durian_dataset = durian_dataset
            self.pv_dataset = pv_dataset
            self.pv_class_offset = pv_class_offset
            
        def __getitem__(self, index):
            if index < len(self.durian_dataset):
                return self.durian_dataset[index]
            else:
                x, y = self.pv_dataset[index - len(self.durian_dataset)]
                return x, y + self.pv_class_offset
        
        def __len__(self):
            return len(self.durian_dataset) + len(self.pv_dataset)
    
    # Create combined datasets
    durian_classes_count = len(durian_train.classes)
    combined_train = CombinedDataset(durian_train, pv_train_transformed, durian_classes_count)
    combined_val = CombinedDataset(durian_val, pv_val_transformed, durian_classes_count)
    
    # Create data loaders
    train_loader = DataLoader(
        combined_train, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    
    val_loader = DataLoader(
        combined_val, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    
    # Summary
    total_classes = durian_classes_count + len(pv_full_dataset.classes)
    all_class_names = durian_train.classes + pv_full_dataset.classes
    
    print(f"✅ Dataset loaded: {len(combined_train):,} train, {len(combined_val):,} val")
    print(f"   Total classes: {total_classes} (Durian: {durian_classes_count}, PlantVillage: {len(pv_full_dataset.classes)})")
    
    return train_loader, val_loader, total_classes, all_class_names

# Load the datasets
train_loader, val_loader, num_classes, class_names = load_combined_dataset()

In [ ]:
# ===================================================================
# DATASET VERIFICATION - CLEANED
# ===================================================================
print("📊 Dataset Summary:")
print(f"   Classes: {num_classes}")
print(f"   Train samples: {len(train_loader.dataset):,}")
print(f"   Val samples: {len(val_loader.dataset):,}")
print(f"   Total: {len(train_loader.dataset) + len(val_loader.dataset):,} samples")
print(f"   Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {batch_size}")
print(f"   Device: {device}")
print(f"✅ Ready for training!")

In [ ]:
# ===================================================================
# SIMPLE DATASET VERIFICATION - NO DUPLICATES
# ===================================================================
from pathlib import Path

def verify_dataset_simple(root_dir, dataset_name):
    """Simple verification without JPG/jpg duplicates"""
    print(f"📊 {dataset_name}:")
    
    root_path = Path(root_dir)
    if not root_path.exists():
        print(f"   ❌ Not found: {root_dir}")
        return 0, 0
    
    total_images = 0
    total_classes = 0
    
    # Check structure
    subdirs = [d for d in root_path.iterdir() if d.is_dir()]
    
    if any(d.name == 'train' for d in subdirs):
        # Pre-split structure (Durian)
        for split in ['train', 'val', 'test']:
            split_dir = root_path / split
            if split_dir.exists():
                classes = [d for d in split_dir.iterdir() if d.is_dir()]
                if split == 'train':
                    total_classes = len(classes)
                
                for class_dir in classes:
                    # Use pathlib.glob with lowercase pattern to avoid duplicates
                    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
                    total_images += len(images)
    else:
        # Direct classes (PlantVillage)
        classes = [d for d in subdirs if d.is_dir()]
        total_classes = len(classes)
        
        for class_dir in classes:
            # Use pathlib.glob with lowercase pattern only
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
            total_images += len(images)
    
    print(f"   📁 Classes: {total_classes}")
    print(f"   🖼️  Images: {total_images:,}")
    return total_images, total_classes

# Quick verification
print("🔍 Dataset Verification:")
print("=" * 50)
durian_images, durian_classes = verify_dataset_simple(DURIAN_DIR, "Durian")
pv_images, pv_classes = verify_dataset_simple(PLANTVILLAGE_DIR, "PlantVillage Color")

total_images = durian_images + pv_images
total_classes = durian_classes + pv_classes

print(f"\n✅ Combined Dataset:")
print(f"   📁 Total Classes: {total_classes}")
print(f"   🖼️  Total Images: {total_images:,}")
print(f"   🎯 Ready for training!")

In [ ]:
# ===================================================================
# MODEL DEFINITION
# ===================================================================
import gc

class ViTClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(ViTClassifier, self).__init__()
        self.vit = vit_b_16(weights='IMAGENET1K_V1' if pretrained else None)
        
        # Replace the classifier head
        in_features = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.vit(x)

# ===================================================================
# MODEL INITIALIZATION
# ===================================================================
model = ViTClassifier(num_classes=num_classes).to(device)

# Verify model
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
output = model(dummy_input)
print("✅ MODEL INITIALIZED:")
print(f"  - Output shape: {output.shape} (expected: [2, {num_classes}])")
print(f"  - Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
del dummy_input, output
gc.collect()

In [ ]:
# ===================================================================
# OPTIMIZER, CRITERION & SCHEDULER SETUP
# ===================================================================
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer - AdamW with weight decay
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler
scheduler = StepLR(optimizer, step_size=7, gamma=0.7)

print("✅ TRAINING COMPONENTS INITIALIZED:")
print(f"  - Criterion: {criterion}")
print(f"  - Optimizer: {optimizer.__class__.__name__} (lr={LEARNING_RATE})")
print(f"  - Scheduler: {scheduler.__class__.__name__} (step_size=7, gamma=0.7)")

In [ ]:
# ===================================================================
# TRAINING FUNCTION
# ===================================================================
def train_model_v2(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, save_path):
    """A more concise and efficient training loop."""
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print("\n🚀 STARTING TRAINING...")
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [T]")
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*train_correct/train_total:.2f}%")

        # Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [V]")
            for inputs, labels in val_bar:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*val_correct/val_total:.2f}%")

        # Record metrics
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_acc = 100 * val_correct / val_total
        epoch_train_loss = train_loss / len(train_loader)
        epoch_val_loss = val_loss / len(val_loader)
        
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)

        # Save best model based on validation accuracy and overfitting gap
        overfitting_gap = epoch_train_acc - epoch_val_acc
        if epoch_val_acc > best_val_acc and (not ENABLE_OVERFITTING_CHECK or overfitting_gap <= MAX_OVERFITTING_GAP):
            best_val_acc = epoch_val_acc
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
                'train_acc': epoch_train_acc,
                'overfitting_gap': overfitting_gap,
                'num_classes': num_classes,
                'classes': classes,
                'dataset_name': DATASET_NAME
            }
            torch.save(checkpoint, save_path)
            print(f"  -> ✅ Best model saved! Val Acc: {best_val_acc:.2f}%, Gap: {overfitting_gap:.2f}%")
        
        scheduler.step()
        print(f"  Epoch {epoch+1:2d}: Train Acc: {epoch_train_acc:.2f}%, Val Acc: {epoch_val_acc:.2f}% | Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    print("\n🏁 TRAINING COMPLETE!")
    print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
    return history

In [ ]:
# Enhanced plotting and model information functions
def plot_training_history(train_losses, train_accs, val_losses, val_accs):
    """
    Enhanced plot training and validation loss and accuracy curves with better visualization
    """
    epochs_range = range(len(train_losses))
    
    plt.figure(figsize=(18, 6))
    
    # Plot Accuracy
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, val_accs, label='Validation Accuracy', marker='o', linewidth=2, color='red')
    plt.plot(epochs_range, train_accs, label='Training Accuracy', marker='s', linewidth=2, color='blue')
    plt.title('Training and Validation Accuracy', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot Loss
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, val_losses, label='Validation Loss', marker='o', linewidth=2, color='red')
    plt.plot(epochs_range, train_losses, label='Training Loss', marker='s', linewidth=2, color='blue')
    plt.title('Training and Validation Loss', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot Best Accuracy Evolution
    plt.subplot(1, 3, 3)
    best_accs = []
    best_so_far = 0
    for acc in val_accs:
        if acc > best_so_far:
            best_so_far = acc
        best_accs.append(best_so_far)
    
    plt.plot(best_accs, label='Best Accuracy So Far', marker='*', linewidth=2, color='green')
    plt.title('Best Accuracy Evolution', fontsize=14)
    plt.xlabel('Epochs')
    plt.ylabel('Best Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    # Print final statistics
    print(f"📈 Training Summary:")
    print(f"   Final Training Accuracy: {train_accs[-1]:.2f}%")
    print(f"   Final Validation Accuracy: {val_accs[-1]:.2f}%")
    print(f"   Best Validation Accuracy: {max(val_accs):.2f}%")
    print(f"   Final Training Loss: {train_losses[-1]:.4f}")
    print(f"   Final Validation Loss: {val_losses[-1]:.4f}")

def load_model_info(model_path):
    """
    Load and display comprehensive model information from checkpoint
    """
    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        return None
    
    try:
        checkpoint = torch.load(model_path, map_location=device)
        
        print(f"📂 Model Information from: {os.path.basename(model_path)}")
        print("=" * 70)
        
        # Training metadata
        print(f"🏋️  Training Metadata:")
        print(f"   Model: {checkpoint.get('model_name', 'Unknown')}")
        print(f"   Dataset: {checkpoint.get('dataset_name', 'Unknown')}")
        print(f"   Combined Dataset: {checkpoint.get('use_combined_dataset', 'Unknown')}")
        print(f"   Training started: {checkpoint.get('start_time', 'Unknown')}")
        print(f"   Model saved: {checkpoint.get('save_time', 'Unknown')}")
        print(f"   Device used: {checkpoint.get('device', 'Unknown')}")
        
        # Performance metrics
        print(f"\n📊 Performance Metrics:")
        print(f"   Best Epoch: {checkpoint.get('best_epoch', 'Unknown')}")
        print(f"   Best Validation Accuracy: {checkpoint.get('best_acc', 0):.4f}")
        print(f"   Final Training Accuracy: {checkpoint.get('train_acc', 0):.4f}")
        print(f"   Final Validation Accuracy: {checkpoint.get('val_acc', 0):.4f}")
        print(f"   Final Training Loss: {checkpoint.get('train_loss', 0):.4f}")
        print(f"   Final Validation Loss: {checkpoint.get('val_loss', 0):.4f}")
        print(f"   Overfitting Gap: {checkpoint.get('overfitting_gap', 0):.4f}")
        
        # Configuration
        print(f"\n⚙️  Configuration:")
        print(f"   Learning Rate: {checkpoint.get('learning_rate', 'Unknown')}")
        print(f"   Batch Size: {checkpoint.get('batch_size', 'Unknown')}")
        print(f"   Image Size: {checkpoint.get('img_size', 'Unknown')}")
        print(f"   Number of Classes: {checkpoint.get('num_classes', 'Unknown')}")
        print(f"   Total Epochs: {checkpoint.get('num_epochs', 'Unknown')}")
        
        # Overfitting detection
        print(f"\n🛡️  Overfitting Detection:")
        print(f"   Enabled: {checkpoint.get('overfitting_check_enabled', 'Unknown')}")
        print(f"   Threshold: {checkpoint.get('overfitting_threshold', 'Unknown')}")
        
        # Dataset information
        if 'dataset_sizes' in checkpoint:
            sizes = checkpoint['dataset_sizes']
            print(f"\n📊 Dataset Information:")
            print(f"   Training samples: {sizes.get('train', 'Unknown')}")
            print(f"   Validation samples: {sizes.get('val', 'Unknown')}")
            print(f"   Test samples: {sizes.get('test', 'Unknown')}")
        
        # Classes
        if 'classes' in checkpoint:
            classes_list = checkpoint['classes']
            print(f"\n🏷️  Classes ({len(classes_list)}):")
            for i, class_name in enumerate(classes_list):
                print(f"   {i:2d}: {class_name}")
        
        print("=" * 70)
        return checkpoint
        
    except Exception as e:
        print(f"❌ Error loading model info: {str(e)}")
        return None

# Function to plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, classes, normalize=False):
    """
    Plot confusion matrix with enhanced visualization
    """
    from sklearn.metrics import confusion_matrix
    import itertools
    
    cm = confusion_matrix(y_true, y_pred)
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = "Normalized Confusion Matrix"
        fmt = '.2f'
    else:
        title = 'Confusion Matrix'
        fmt = 'd'
    
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title, fontsize=16)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label', fontsize=14)
    plt.xlabel('Predicted label', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Enhanced evaluation function with progress bars
def evaluate_model(model, test_loader, criterion, device, classes):
    """
    Evaluate model with progress bar and detailed metrics
    """
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_targets = []
    class_correct = list(0. for i in range(len(classes)))
    class_total = list(0. for i in range(len(classes)))
    
    print("🔍 Evaluating model...")
    
    # Progress bar for evaluation
    eval_pbar = tqdm(test_loader, desc="📊 Testing", unit="batch")
    
    with torch.no_grad():
        for data, target in eval_pbar:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item()
            
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            # Collect all predictions and targets
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            
            # Per-class accuracy calculation
            c = (predicted == target).squeeze()
            for i in range(target.size(0)):
                label = target[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1
            
            # Update progress bar
            current_acc = 100. * correct / total
            eval_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{current_acc:.2f}%'
            })
    
    eval_pbar.close()
    
    # Calculate final metrics
    test_loss /= len(test_loader)
    test_acc = 100. * correct / total
    
    print(f"\n📈 Overall Test Results:")
    print(f"   🎯 Test Loss: {test_loss:.4f}")
    print(f"   📊 Test Accuracy: {test_acc:.2f}% ({correct}/{total})")
    
    # Per-class accuracy
    print(f"\n📋 Per-Class Accuracy:")
    for i, class_name in enumerate(classes):
        if class_total[i] > 0:
            class_acc = 100. * class_correct[i] / class_total[i]
            print(f"   {class_name:25s}: {class_acc:6.2f}% ({int(class_correct[i])}/{int(class_total[i])})")
        else:
            print(f"   {class_name:25s}: No samples")
    
    return test_loss, test_acc, np.array(all_predictions), np.array(all_targets)

# Function to run complete evaluation with visualizations
def complete_evaluation(model, test_loader, criterion, device, classes):
    """
    Run complete evaluation with progress bars and visualizations
    """
    print("🚀 Starting Complete Model Evaluation")
    print("=" * 60)
    
    # Evaluate model
    test_loss, test_acc, y_pred, y_true = evaluate_model(model, test_loader, criterion, device, classes)
    
    # Classification report
    print(f"\n📊 Detailed Classification Report:")
    print("=" * 60)
    from sklearn.metrics import classification_report
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Confusion matrices
    print(f"\n📊 Confusion Matrix Visualization:")
    plot_confusion_matrix(y_true, y_pred, classes, normalize=False)
    plot_confusion_matrix(y_true, y_pred, classes, normalize=True)
    
    return test_loss, test_acc, y_pred, y_true

In [ ]:
# ===================================================================
# OPTIMIZED TRAINING LOOP - SIMPLIFIED AND FAST
# ===================================================================
import time
from tqdm import tqdm
import gc

# Training configuration - using variables from config cell
patience = 5
early_stop_counter = 0
best_val_acc = 0.0

# Training metrics storage
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print(f"🚀 Starting ViT training on {num_classes} classes...")
print(f"📅 Epochs: {epochs} | Batch Size: {batch_size} | Patience: {patience}")
print(f"🛡️  Overfitting Protection: Max gap {MAX_OVERFITTING_GAP*100:.1f}%")
print(f"💾 Model will be saved to: {MODEL_SAVE_PATH}")
print("-" * 70)

start_time = time.time()

for epoch in range(epochs):
    epoch_start = time.time()
    
    print(f"\n🔄 Epoch {epoch+1}/{epochs}")
    
    # ===== TRAINING PHASE =====
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    # Single progress bar for training
    with tqdm(train_loader, desc="📚 Training", unit="batch", leave=False) as pbar:
        for batch_idx, (data, target) in enumerate(pbar):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            # Update metrics
            train_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            train_total += target.size(0)
            train_correct += (predicted == target).sum().item()
            
            # Update progress bar every few batches
            if batch_idx % 10 == 0:
                current_loss = train_loss / (batch_idx + 1)
                current_acc = 100. * train_correct / train_total
                pbar.set_postfix({
                    'Loss': f'{current_loss:.4f}',
                    'Acc': f'{current_acc:.2f}%'
                })
    
    # Calculate epoch training metrics
    epoch_train_loss = train_loss / len(train_loader)
    epoch_train_acc = 100. * train_correct / train_total
    
    # ===== VALIDATION PHASE =====
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        with tqdm(val_loader, desc="🔍 Validation", unit="batch", leave=False) as pbar:
            for batch_idx, (data, target) in enumerate(pbar):
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                
                val_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                val_total += target.size(0)
                val_correct += (predicted == target).sum().item()
                
                # Update progress bar every few batches
                if batch_idx % 5 == 0:
                    current_loss = val_loss / (batch_idx + 1)
                    current_acc = 100. * val_correct / val_total
                    pbar.set_postfix({
                        'Loss': f'{current_loss:.4f}',
                        'Acc': f'{current_acc:.2f}%'
                    })
    
    # Calculate epoch validation metrics
    epoch_val_loss = val_loss / len(val_loader)
    epoch_val_acc = 100. * val_correct / val_total
    
    # Store metrics
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    
    # Calculate overfitting gap
    overfitting_gap = (epoch_train_acc - epoch_val_acc) / 100  # Convert to decimal
    gap_status = "✅" if overfitting_gap <= MAX_OVERFITTING_GAP else "⚠️"
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Calculate timing
    epoch_duration = time.time() - epoch_start
    remaining_epochs = epochs - (epoch + 1)
    eta = epoch_duration * remaining_epochs
    
    # Print epoch summary
    print(f"   📊 Train: Loss {epoch_train_loss:.4f} | Acc {epoch_train_acc:.2f}%")
    print(f"   📊 Val:   Loss {epoch_val_loss:.4f} | Acc {epoch_val_acc:.2f}%")
    print(f"   {gap_status} Gap: {overfitting_gap:.3f} | ⏱️ {epoch_duration:.1f}s | ETA: {eta/60:.1f}min | LR: {current_lr:.6f}")
    
    # ===== MODEL SAVING WITH OVERFITTING PROTECTION =====
    save_model = False
    
    if ENABLE_OVERFITTING_CHECK:
        if overfitting_gap <= MAX_OVERFITTING_GAP:
            if epoch_val_acc > best_val_acc:
                best_val_acc = epoch_val_acc
                early_stop_counter = 0
                save_model = True
                print(f"     💾 New best model! Val Acc: {epoch_val_acc:.2f}%")
            else:
                early_stop_counter += 1
                print(f"     ⏰ No improvement for {early_stop_counter} epochs")
        else:
            early_stop_counter += 1
            print(f"     ⚠️ Overfitting detected! Gap: {overfitting_gap:.3f} > {MAX_OVERFITTING_GAP}")
    else:
        # No overfitting check - save if validation accuracy improves
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            early_stop_counter = 0
            save_model = True
            print(f"     💾 New best model! Val Acc: {epoch_val_acc:.2f}%")
        else:
            early_stop_counter += 1
            print(f"     ⏰ No improvement for {early_stop_counter} epochs")
    
    # Save model if needed
    if save_model:
        # Calculate dataset sizes for checkpoint
        train_size = len(train_loader.dataset)
        val_size = len(val_loader.dataset)
        dataset_sizes = {
            'train': train_size,
            'val': val_size,
            'test': 0  # No test set during training
        }
        
        checkpoint = {
            'epoch': epoch + 1,
            'epochs': epochs,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'train_losses': train_losses,
            'train_accs': train_accs,
            'val_losses': val_losses,
            'val_accs': val_accs,
            'model_name': MODEL_NAME,
            'num_classes': num_classes,
            'classes': class_names,
            'dataset_sizes': dataset_sizes,
            'dataset_name': DATASET_NAME,
            'learning_rate': LEARNING_RATE,
            'batch_size': batch_size,
            'img_size': IMG_SIZE,
            'overfitting_gap': overfitting_gap,
            'overfitting_check_enabled': ENABLE_OVERFITTING_CHECK,
            'overfitting_threshold': MAX_OVERFITTING_GAP
        }
        
        torch.save(checkpoint, MODEL_SAVE_PATH)
    
    # Early stopping check
    if early_stop_counter >= patience:
        print(f"\n🛑 Early stopping triggered! No improvement for {patience} epochs.")
        break
    
    # Memory cleanup
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ===== TRAINING COMPLETED =====
end_time = time.time()
total_time = end_time - start_time

print(f"\n🎉 Training completed!")
print(f"   ⏱️  Total time: {total_time/60:.1f} minutes ({total_time:.1f} seconds)")
print(f"   🏆 Best validation accuracy: {best_val_acc:.2f}%")
print(f"   📊 Final overfitting gap: {overfitting_gap:.3f}")
print(f"   📈 Epochs completed: {len(train_accs)}/{epochs}")
print(f"   💾 Best model saved to: {MODEL_SAVE_PATH}")

# Store training history for plotting
training_history = {
    'train_losses': train_losses,
    'train_accs': train_accs,
    'val_losses': val_losses,
    'val_accs': val_accs
}

print(f"\n✅ Training variables ready for plotting and analysis!")

# Compare Models

In [ ]:
# ===================================================================
# TRAINING VISUALIZATION
# ===================================================================
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(train_history['loss'], label='Train Loss', color='blue', linewidth=2)
ax1.plot(val_history['loss'], label='Val Loss', color='red', linewidth=2)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(train_history['acc'], label='Train Acc', color='blue', linewidth=2)
ax2.plot(val_history['acc'], label='Val Acc', color='red', linewidth=2)
ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final summary
print(f"📊 Final Training Summary:")
print(f"   Best validation accuracy: {best_val_acc:.2f}%")
print(f"   Final train accuracy: {train_history['acc'][-1]:.2f}%")
print(f"   Total epochs trained: {len(train_history['acc'])}")
print(f"   Model saved as: best_vit_model.pth")

In [ ]:
# Comprehensive evaluation with enhanced visualizations and detailed analysis
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

if 'model' in locals() and 'test_results' in locals():
    print("🔍 Running Comprehensive Enhanced Evaluation")
    print("=" * 70)
    
    # Extract results
    y_true = test_results['y_true']
    y_pred = test_results['y_pred']
    
    # Detailed classification report
    print(f"📊 Detailed Classification Report:")
    print("=" * 70)
    print(classification_report(y_true, y_pred, target_names=classes, digits=4))
    
    # Confusion matrices with enhanced visualization
    print(f"\n📊 Confusion Matrix Visualizations:")
    
    # Raw counts confusion matrix
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    cm_raw = confusion_matrix(y_true, y_pred)
    plt.imshow(cm_raw, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix (Raw Counts)', fontsize=14)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, [c[:15] + '...' if len(c) > 15 else c for c in classes], rotation=45, ha='right')
    plt.yticks(tick_marks, [c[:15] + '...' if len(c) > 15 else c for c in classes])
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    
    # Normalized confusion matrix
    plt.subplot(1, 2, 2)
    cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
    plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix (Normalized)', fontsize=14)
    plt.colorbar()
    plt.xticks(tick_marks, [c[:15] + '...' if len(c) > 15 else c for c in classes], rotation=45, ha='right')
    plt.yticks(tick_marks, [c[:15] + '...' if len(c) > 15 else c for c in classes])
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Model performance summary
    print(f"\n📈 Enhanced Model Performance Summary:")
    print("=" * 70)
    
    if 'checkpoint' in locals() and checkpoint:
        print(f"🏋️  Training Performance:")
        print(f"   Best Epoch: {checkpoint.get('best_epoch', 'Unknown')}")
        print(f"   Training Accuracy: {checkpoint.get('train_acc', 0):.4f}")
        print(f"   Validation Accuracy: {checkpoint.get('best_acc', 0):.4f}")
        print(f"   Overfitting Gap: {checkpoint.get('overfitting_gap', 0):.4f}")
        
        print(f"\n🧪 Test Performance:")
        print(f"   Test Accuracy: {test_results['test_acc']:.2f}%")
        print(f"   Test Loss: {test_results['test_loss']:.4f}")
        
        print(f"\n⚖️  Generalization Analysis:")
        val_acc = checkpoint.get('best_acc', 0)
        test_acc = test_results['test_acc'] / 100
        train_acc = checkpoint.get('train_acc', 0)
        
        print(f"   Train vs Val Gap: {train_acc - val_acc:+.4f}")
        print(f"   Val vs Test Gap: {val_acc - test_acc:+.4f}")
        print(f"   Train vs Test Gap: {train_acc - test_acc:+.4f}")
        
        # Performance classification
        if abs(val_acc - test_acc) < 0.02:
            generalization = "🟢 Excellent generalization"
        elif abs(val_acc - test_acc) < 0.05:
            generalization = "🟡 Good generalization"
        else:
            generalization = "🔴 Poor generalization"
        
        print(f"   Generalization: {generalization}")
        
        print(f"\n📊 Dataset Information:")
        if 'dataset_sizes' in checkpoint:
            sizes = checkpoint['dataset_sizes']
            total_samples = sum(sizes.values()) if sizes else 0
            print(f"   Total Samples: {total_samples:,}")
            print(f"   Training: {sizes.get('train', 0):,} ({100*sizes.get('train', 0)/total_samples:.1f}%)")
            print(f"   Validation: {sizes.get('val', 0):,} ({100*sizes.get('val', 0)/total_samples:.1f}%)")
            print(f"   Test: {sizes.get('test', 0):,} ({100*sizes.get('test', 0)/total_samples:.1f}%)")
        
        print(f"   Classes: {len(classes)}")
        print(f"   Dataset Type: {checkpoint.get('dataset_name', 'Unknown')}")
    
    print("=" * 70)
    
else:
    print("❌ Model or test results not available.")
    print("   Please run the previous evaluation cell first.")

In [ ]:
# Enhanced training history visualization with checkpoint data
print("📊 Enhanced Training History Visualization")
print("=" * 60)

# Try to get training history from checkpoint first, then from local variables
history_source = "unknown"
train_losses_data = None
train_accs_data = None
val_losses_data = None
val_accs_data = None

# Check checkpoint data first
if 'checkpoint' in locals() and checkpoint:
    if 'train_losses' in checkpoint:
        train_losses_data = checkpoint['train_losses']
        train_accs_data = checkpoint['train_accs']
        val_losses_data = checkpoint['val_losses']
        val_accs_data = checkpoint['val_accs']
        history_source = "checkpoint"
        print(f"📂 Using training history from saved checkpoint")

# Check local training variables
elif 'training_history' in locals():
    train_losses_data = training_history['train_losses']
    train_accs_data = training_history['train_accs']
    val_losses_data = training_history['val_losses']
    val_accs_data = training_history['val_accs']
    history_source = "local_training"
    print(f"🔄 Using training history from recent training session")

# Check individual variables
elif 'train_losses' in locals() and 'val_losses' in locals():
    train_losses_data = train_losses
    train_accs_data = train_accs
    val_losses_data = val_losses
    val_accs_data = val_accs
    history_source = "local_variables"
    print(f"? Using training history from local variables")

# Display training history if available
if train_losses_data is not None:
    print(f"📈 Plotting training history (source: {history_source})")
    
    # Use enhanced plotting function
    plot_training_history(train_losses_data, train_accs_data, val_losses_data, val_accs_data)
    
    # Additional detailed statistics
    print(f"\n? Detailed Training Statistics:")
    print(f"   Total Epochs Trained: {len(train_accs_data)}")
    print(f"   Best Training Accuracy: {max(train_accs_data):.4f} (Epoch {np.argmax(train_accs_data) + 1})")
    print(f"   Best Validation Accuracy: {max(val_accs_data):.4f} (Epoch {np.argmax(val_accs_data) + 1})")
    print(f"   Final Training Accuracy: {train_accs_data[-1]:.4f}")
    print(f"   Final Validation Accuracy: {val_accs_data[-1]:.4f}")
    print(f"   Final Training Loss: {train_losses_data[-1]:.4f}")
    print(f"   Final Validation Loss: {val_losses_data[-1]:.4f}")
    
    # Overfitting analysis
    final_gap = train_accs_data[-1] - val_accs_data[-1]
    best_train_idx = np.argmax(train_accs_data)
    best_val_idx = np.argmax(val_accs_data)
    
    print(f"\n🔍 Overfitting Analysis:")
    print(f"   Final Overfitting Gap: {final_gap:+.4f}")
    print(f"   Best Train Accuracy at Epoch: {best_train_idx + 1}")
    print(f"   Best Val Accuracy at Epoch: {best_val_idx + 1}")
    
    if final_gap > 0.05:
        print(f"   ⚠️  Warning: Potential overfitting detected (gap > 0.05)")
    elif final_gap > 0.02:
        print(f"   🟡 Caution: Moderate training-validation gap")
    else:
        print(f"   ✅ Good training-validation balance")
    
    # Learning curve analysis
    if len(train_accs_data) >= 10:
        recent_epochs = 5
        recent_train_trend = np.mean(train_accs_data[-recent_epochs:]) - np.mean(train_accs_data[-2*recent_epochs:-recent_epochs])
        recent_val_trend = np.mean(val_accs_data[-recent_epochs:]) - np.mean(val_accs_data[-2*recent_epochs:-recent_epochs])
        
        print(f"\n📈 Learning Curve Analysis (last {recent_epochs} epochs):")
        print(f"   Training Accuracy Trend: {recent_train_trend:+.4f}")
        print(f"   Validation Accuracy Trend: {recent_val_trend:+.4f}")
        
        if recent_val_trend > 0.01:
            print(f"   📈 Model still improving")
        elif recent_val_trend > -0.01:
            print(f"   📊 Model has converged")
        else:
            print(f"   📉 Model may be degrading")

else:
    print("❌ Training history not available.")
    print("   This could happen if:")
    print("   - Model was loaded from checkpoint without training history")
    print("   - Training was not completed in this session")
    print("   - Training variables were cleared")
    print("\n💡 To see training history:")
    print("   - Run the training cell to generate new history")
    print("   - Or check if training history is saved in the model checkpoint")

# Make Prediction

In [ ]:
# Enhanced prediction with better visualization and model information
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as transforms

# Enhanced prediction function
def predict_image_enhanced(image_path, model, classes, device, show_top_k=5):
    """
    Enhanced prediction function with detailed analysis
    """
    if not os.path.exists(image_path):
        print(f"❌ Image not found: {image_path}")
        return None
    
    # Load and preprocess image
    img_rgb = Image.open(image_path).convert('RGB')
    
    # Use the same transform as validation
    single_image_transform = transforms.Compose([
        transforms.Resize(int(IMG_SIZE / 0.875)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    img_tensor = single_image_transform(img_rgb).unsqueeze(0).to(device)
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        top_probs, top_indices = torch.topk(probabilities[0], min(show_top_k, len(classes)))
    
    # Display image and results
    plt.figure(figsize=(15, 6))
    
    # Image display
    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.title(f"Input Image\n{os.path.basename(image_path)}", fontsize=14)
    plt.axis('off')
    
    # Prediction results
    plt.subplot(1, 2, 2)
    class_names = [classes[idx] for idx in top_indices]
    confidences = [prob.item() for prob in top_probs]
    
    # Create horizontal bar chart
    y_pos = np.arange(len(class_names))
    colors = ['green' if i == 0 else 'lightblue' for i in range(len(class_names))]
    
    plt.barh(y_pos, confidences, color=colors)
    plt.yticks(y_pos, [name[:20] + '...' if len(name) > 20 else name for name in class_names])
    plt.xlabel('Confidence Score')
    plt.title(f'Top {len(class_names)} Predictions', fontsize=14)
    plt.xlim(0, 1)
    
    # Add confidence values on bars
    for i, conf in enumerate(confidences):
        plt.text(conf + 0.01, i, f'{conf:.3f}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    predicted_class = class_names[0]
    confidence = confidences[0]
    
    print(f"🖼️  Image Analysis Results:")
    print(f"   📁 File: {os.path.basename(image_path)}")
    print(f"   📏 Size: {img_rgb.size}")
    print(f"   🏷️  Top Prediction: {predicted_class}")
    print(f"   📊 Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
    
    print(f"\n📋 Top {len(class_names)} Predictions:")
    for i, (class_name, conf) in enumerate(zip(class_names, confidences)):
        status = "🥇" if i == 0 else f"{i+1:2d}."
        print(f"   {status} {class_name:30s}: {conf:.4f} ({conf*100:.2f}%)")
    
    # Confidence analysis
    print(f"\n🔍 Prediction Analysis:")
    if confidence > 0.9:
        analysis = "🟢 Very confident prediction"
    elif confidence > 0.7:
        analysis = "🟡 Confident prediction"
    elif confidence > 0.5:
        analysis = "🟠 Moderate confidence"
    else:
        analysis = "🔴 Low confidence prediction"
    
    print(f"   {analysis}")
    
    # Check if prediction is ambiguous
    if len(confidences) > 1:
        confidence_gap = confidences[0] - confidences[1]
        print(f"   Gap to 2nd prediction: {confidence_gap:.4f}")
        
        if confidence_gap < 0.1:
            print(f"   ⚠️  Warning: Close competition with {class_names[1]}")
        elif confidence_gap < 0.2:
            print(f"   🟡 Moderate separation from {class_names[1]}")
        else:
            print(f"   ✅ Clear distinction from other classes")
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence,
        'top_predictions': list(zip(class_names, confidences)),
        'image_path': image_path
    }

# Test with sample images
test_images = [
    "./realImage/late_blight_tomato_leaf5x12001-1.jpg",
    "./realImage/UK_advice-pests-diseases-tomato-leaf-mould_main.jpg"
]

if 'model' in locals():
    print("🔍 Enhanced Image Prediction Testing")
    print("=" * 60)
    
    for i, image_path in enumerate(test_images):
        print(f"\n🖼️  Test Image {i+1}:")
        print("-" * 40)
        
        if os.path.exists(image_path):
            result = predict_image_enhanced(image_path, model, classes, device, show_top_k=3)
            
            if result:
                print(f"✅ Prediction completed successfully")
            else:
                print(f"❌ Prediction failed")
        else:
            print(f"⚠️  Image not found: {image_path}")
            
            # List available images for reference
            realimage_dir = "./realImage"
            if os.path.exists(realimage_dir):
                available_images = [f for f in os.listdir(realimage_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                if available_images:
                    print(f"\n📁 Available images in {realimage_dir}:")
                    for img in available_images[:5]:  # Show first 5
                        print(f"   - {img}")
                    if len(available_images) > 5:
                        print(f"   ... and {len(available_images) - 5} more")
    
    # Instructions for custom prediction
    print(f"\n💡 To test with your own image:")
    print(f"   1. Place your image in the ./realImage/ folder")
    print(f"   2. Update the image_path variable above")
    print(f"   3. Run the prediction cell again")
    
else:
    print("❌ Model not loaded. Please run the model loading/training cell first.")